In [10]:
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", False)
import os, json, numpy as np, matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts/experiments_large_scale/')

import run_LowRank

from run_LowRank import (seed_everything, get_device,
            stratified_equal_halves
)

In [11]:
"""
Timepoints, indexed by AnnData index
"""
timepoints_across_datasets =  [['E8.5', 'E8.75', 'E9.0', 'E9.25', 'E9.5', 'E9.75', 'E10.0']]
'''
timepoints_across_datasets =  [['E8.5', 'E8.75', 'E9.0', 'E9.25', 'E9.5', 'E9.75', 'E10.0', 'E10.25', 'E10.5', 'E10.75'], \
                               ['E11.0', 'E11.25', 'E11.5', 'E11.75', 'E12.0', 'E12.25', 'E12.5', 'E12.75', 'E13.0', 'E13.25', 'E13.5', 'E13.75'], \
                               ['E14.0', 'E14.25', 'E14.333', 'E14.75', 'E15.0', 'E15.25', 'E15.5', 'E15.75', 'E16.0', 'E16.25', 'E16.5', 'E16.75'], \
                               ['E17.0', 'E17.25', 'E17.5', 'E17.75', 'E18.0', 'E18.25', 'E18.5', 'E18.75']]
'''

"""
For simplicity, we choose the first replicate for each timepoint as our dataset.
"""
replicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30']]
'''
replicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30', 'embryo_33', 'embryo_34', 'embryo_35'], \
                              ['embryo_36', 'embryo_37', 'embryo_38', 'embryo_39', 'embryo_40', 'embryo_41', 'embryo_42', 'embryo_43', 'embryo_45', 'embryo_47', 'embryo_49', 'embryo_52'], \
                              ['embryo_53', 'embryo_54', 'embryo_55', 'embryo_56', 'embryo_57', 'embryo_58', 'embryo_59', 'embryo_60', 'embryo_61', 'embryo_62', 'embryo_63', 'embryo_64'], \
                              ['embryo_65', 'embryo_66', 'embryo_67', 'embryo_68', 'embryo_69', 'embryo_70', 'embryo_71', 'embryo_72']]
'''


"\nreplicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30', 'embryo_33', 'embryo_34', 'embryo_35'],                               ['embryo_36', 'embryo_37', 'embryo_38', 'embryo_39', 'embryo_40', 'embryo_41', 'embryo_42', 'embryo_43', 'embryo_45', 'embryo_47', 'embryo_49', 'embryo_52'],                               ['embryo_53', 'embryo_54', 'embryo_55', 'embryo_56', 'embryo_57', 'embryo_58', 'embryo_59', 'embryo_60', 'embryo_61', 'embryo_62', 'embryo_63', 'embryo_64'],                               ['embryo_65', 'embryo_66', 'embryo_67', 'embryo_68', 'embryo_69', 'embryo_70', 'embryo_71', 'embryo_72']]\n"

In [ ]:
import os, json, sys, gc
import numpy as np
import pandas as pd
import scanpy as sc
import single_cell
import importlib

importlib.reload(single_cell)
importlib.reload(run_LowRank)

# Cell-type labels (index must contain 'cell_id' to join on)
df_cell = pd.read_csv("/scratch/gpfs/ph3641/hm_ot/df_cell.csv").set_index("cell_id")
# Whether minor subsampling is needed -- for HiRef to work need many divisors, so decrease n slightly to get this
subsample_to_nonprime_tp = [False, True, True, True, True, True, True]

all_results = []

for i in range(len(timepoints_across_datasets)):
    # --- load dataset i ---
    filehandle_ME = f"/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{i+1}.h5ad"
    adata = sc.read_h5ad(filehandle_ME, backed="r")

    timepoints = timepoints_across_datasets[i]
    replicates = replicates_across_datasets[i]

    for tp in range(len(timepoints)):
        subset_adata = None
        
        if (tp + 1) < len(timepoints):
            # pair within the same dataset
            t1, t2 = timepoints[tp], timepoints[tp + 1]
            r1, r2 = replicates[tp], replicates[tp + 1]
            print(f"[intra] Aligning timepoint {t1} ({r1}) → {t2} ({r2})")

            # subset by original string fields; move to memory
            subset_adata = adata[
                (adata.obs["day"].isin([t1, t2])) &
                (adata.obs["embryo_id"].isin([r1, r2]))
            ].to_memory()

        elif (i + 1) < len(timepoints_across_datasets):
            # cross to the next dataset's first timepoint
            timepoints2 = timepoints_across_datasets[i + 1]
            replicates2 = replicates_across_datasets[i + 1]
            t1, t2 = timepoints[tp], timepoints2[0]
            r1, r2 = replicates[tp], replicates2[0]
            print(f"[inter] Aligning timepoint {t1} ({r1}) → {t2} ({r2})")

            # last slice from current dataset to memory
            adata1 = adata[
                (adata.obs["day"] == t1) &
                (adata.obs["embryo_id"] == r1)
            ].to_memory()

            # open next dataset and slice first pair to memory
            filehandle_ME2 = f"/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{i+2}.h5ad"
            adata2_full = sc.read_h5ad(filehandle_ME2, backed="r")
            adata2 = adata2_full[
                (adata2_full.obs["day"] == t2) &
                (adata2_full.obs["embryo_id"] == r2)
            ].to_memory()
            del adata2_full; gc.collect()

            # concatenate the two to one in-memory object
            subset_adata = adata1.concatenate(adata2, index_unique=None)
            del adata1, adata2; gc.collect()
        else:
            # nothing to align at the tail
            continue

        # skip if empty (can happen with missing replicates)
        if subset_adata.n_obs == 0:
            print("  -> empty subset, skipping")
            continue

        # integrate labels (expects 'cell_id' present as a column)
        if "cell_id" not in subset_adata.obs.columns:
            print("  -> missing 'cell_id' in obs; cannot join labels, skipping")
            del subset_adata; gc.collect()
            continue

        subset_adata.obs = subset_adata.obs.set_index("cell_id", drop=False)
        subset_adata.obs = subset_adata.obs.join(df_cell[["celltype_update"]], how="left")

        # convert 'day' to ordered numeric category (for plotting/metadata), AFTER subsetting
        # keep original strings for selection logic above
        try:
            day_num = subset_adata.obs["day"].astype(str).str.extract(r"(\d+(?:\.\d+)?)").astype(float)[0]
            subset_adata.obs["day_num"] = day_num
            subset_adata.obs["day_num"] = pd.Categorical(subset_adata.obs["day_num"], ordered=True)
        except Exception as e:
            print(f"  -> warning: could not parse day to numeric: {e}")

        # --- run LR-OT methods + evaluation ---
        try:
            df_out = single_cell.run_pairwise_eval(
                subset_adata, t1, t2,
                methods=( "monge_conj", "frlc", "lot" ),
                label_col="celltype_update",
                seed=0, use_pca=True, pca_key="X_pca", n_comps=50,
                max_rank=None,
                subsample_to_nonprime=subsample_to_nonprime_tp[tp]
                # or set to a fixed rank, e.g., 10
            )
            print(df_out)
            all_results.append(df_out)
        except Exception as e:
            print(f"  -> evaluation failed: {e}")

        # free memory aggressively
        del subset_adata; gc.collect()

    # close backed AnnData view (let file handle go)
    del adata; gc.collect()

# save combined results
if all_results:
    all_results = pd.concat(all_results, ignore_index=True)
    save_csv = "sc_timepair_lr_ot_results.csv"
    all_results.to_csv(save_csv, index=False)
    print("Saved:", save_csv)
else:
    print("No results produced.")


[intra] Aligning timepoint E8.5 (embryo_11) → E8.75 (embryo_14)
Optimized rank-annealing schedule: [27, 697]


2025-09-20 20:08:45.925 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:178 - Initialization Costs: (0.23392217104041718, 0.2346333252575175)


Iteration: 0
  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0        E8.5       E8.75  monge_conj    43  0.504971  0.638494  0.329298   
1        E8.5       E8.75        frlc    43  0.552525  0.556304  0.217489   
2        E8.5       E8.75         lot    43  0.520181  0.604897  0.282690   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  
0  0.617390  0.308064  0.724702    52.148965    18819  
1  0.531313  0.199441  0.525272    12.853811    18819  
2  0.591938  0.271684  0.610776     4.796343    18819  
[intra] Aligning timepoint E8.75 (embryo_14) → E9.0 (embryo_16)
Subsampled to n=30240 from n=30655 [needed for Hierarchical Refinement]
Optimized rank-annealing schedule: [32, 945]


2025-09-20 20:11:49.052 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:178 - Initialization Costs: (0.1982711635732212, 0.19879788299976103)


Iteration: 0
  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0       E8.75        E9.0  monge_conj    53  0.382594  0.588512  0.226170   
1       E8.75        E9.0        frlc    53  0.404844  0.533736  0.174061   
2       E8.75        E9.0         lot    53  0.390198  0.559111  0.193351   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  
0  0.592119  0.226317  0.529787    85.896482    30240  
1  0.541283  0.177729  0.491635    13.289999    30240  
2  0.566558  0.197148  0.486512     6.206905    30240  
[intra] Aligning timepoint E9.0 (embryo_16) → E9.25 (embryo_20)
Subsampled to n=45360 from n=45685 [needed for Hierarchical Refinement]
Optimized rank-annealing schedule: [48, 945]


2025-09-20 20:16:07.128 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:178 - Initialization Costs: (0.222251314270986, 0.2207551161172815)


Iteration: 0


2025-09-20 20:16:45.319465: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 12.78GiB (13718388633 bytes) by rematerialization; only reduced to 15.33GiB (16460236800 bytes), down from 15.33GiB (16460236800 bytes) originally


  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0        E9.0       E9.25  monge_conj    57  0.456009  0.548270  0.183616   
1        E9.0       E9.25        frlc    57  0.481351  0.524186  0.158253   
2        E9.0       E9.25         lot    57  0.453066  0.564528  0.197511   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  
0  0.538051  0.179385  0.470662   141.507670    45360  
1  0.514501  0.155378  0.470641    16.630423    45360  
2  0.552575  0.194443  0.556928    12.315928    45360  
[intra] Aligning timepoint E9.25 (embryo_20) → E9.5 (embryo_24)


In [ ]:
# Cell-type labels for timepoints + replicates
df_cell = pd.read_csv("/scratch/gpfs/ph3641/hm_ot/df_cell.csv")
df_cell = df_cell.set_index("cell_id")

import scanpy as sc

for i in range(len(timepoints_across_datasets)):
    """
    Loading file / AnnData
    """
    filehandle_ME = f'/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{idxs[i]}.h5ad'
    sys.path.insert(0, filehandle_ME)
    adata = sc.read_h5ad(filehandle_ME, backed="r")
    
    timepoints = timepoints_across_datasets[i]
    replicates = replicates_across_datasets[i]
    
    for tp in range(len(timepoints)):
        """
        Iterating across all timepoint pairs, stitching separate AnnDatas together when necessary.
        """
        if (tp+1) < len(timepoints):
            # Aligning two timepoints within a dataset.
            t1 = timepoints[tp]
            t2 = timepoints[tp+1]
            
            r1 = replicates[tp]
            r2 = replicates[tp+1]
            
            print(f'Aligning timepoint {t1} to {t2}')
            
            # Generate a pairwise subset of the dataset
            subset_adata = adata[
                    (adata.obs['day'].isin(timepoints[tp:tp+2])) & 
                    (adata.obs['embryo_id'].isin(replicates[tp:tp+2]))
                ]
        
        elif ( i+1 < len(idxs)-1 ):
            
            timepoints2 = timepoints_across_datasets[i+1]
            replicates2 = replicates_across_datasets[i+1]
            
            # Stitch timepoints between datasets until no remaining AnnData objects left!
            t1 = timepoints[tp]
            t2 = timepoints2[0]
            
            r1 = replicates[tp]
            r2 = replicates2[0]
            
            print(f'Aligning timepoint {t1} to {t2}')
            
            filehandle_ME2 = f'/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{idxs[i]+1}.h5ad'
            sys.path.insert(0, filehandle_ME2)
            
            # Being memory-efficient; save last timepoint into its own object, delete, and load next.
            adata = adata[
                    ( adata.obs['day'] == t1 ) & 
                    ( adata.obs['embryo_id'] == r1 )
                ].to_memory() #.copy(filename='/scratch/gpfs/ph3641/hm_ot/adata_fin.h5ad')
            
            adata2 = sc.read_h5ad(filehandle_ME2, backed="r")

            print(f'time2: {t2}, rep2: {r2}')
            
            adata2 = adata2[
                    (adata2.obs['day'] == t2 ) & 
                    (adata2.obs['embryo_id'] == r2 )
                ].to_memory() #.copy(filename='/scratch/gpfs/ph3641/hm_ot/adata2_init.h5ad')
            
            # Concatenate the two AnnData objects
            subset_adata = adata.concatenate(
                adata2,
                index_unique=None
            )
        
        #if not os.path.isfile( os.path.join(diffmap_dir, t1 + "_" + t2 + "_T.npy") ):
        
        """
        Now, (1) integrate cell-type labels and (2) perform Low Rank OT.
        """
        subset_adata = subset_adata.to_memory()
        
        subset_adata.obs = subset_adata.obs.set_index("cell_id")
        subset_adata.obs = subset_adata.obs.join(df_cell[['celltype_update']], how="left")
        
        subset_adata.obs['day'] = subset_adata.obs['day'].str.extract(r'(\d+(?:\.\d+)?)').astype(float)
        subset_adata.obs['day'] = pd.Categorical(subset_adata.obs['day'], ordered=True)
        
        df_out = run_pairwise_eval(
            subset_adata, t1, t2,
            methods=("monge_conj","lot","frlc"),
            label_col="celltype_update",
            seed=0, use_pca=True, pca_key="X_pca", n_comps=50,
            max_rank=None  # or set to e.g. 10
        )
        
        all_results = [] if 'all_results' not in globals() else all_results
        all_results.append(df_out)
        

In [ ]:
all_results = pd.concat(all_results, ignore_index=True)
save_csv = "/scratch/gpfs/ph3641/hm_ot/sc_timepair_lr_ot_results.csv"
all_results.to_csv(save_csv, index=False)
print("Saved:", save_csv)

In [ ]:


cfg = {
            "root": "/scratch/gpfs/DATASETS/cifar",
            "weights_path": "/home/ph3641/OptimalLowRank/Monge_Conjugation/models/resnet18-f37072fd.pth",
            "device": "cuda",     # "cpu" or "cuda"
            "batch_size": 256,
            "num_workers": 4,
            "model": "resnet18",  # "resnet18", "resnet50", "inception_v3"
            "seed": 123,
            "reg": 0.05,          # Sinkhorn entropic regularization
            "rank": 10,           # rank
            "reuse_embeddings": "",  # path to .npz if you want to reuse
            "save_embeddings": True,
            "run_methods": ["lr_monge"],  # choose which to run
        }

seed_everything(cfg["seed"])
device = get_device(cfg["device"])


A_idx, B_idx = stratified_equal_halves(labels, seed=cfg["seed"])
XA, yA = feats[A_idx], labels[A_idx]
YB, yB = feats[B_idx], labels[B_idx]
print(f"Split sizes: A={XA.shape[0]}, B={YB.shape[0]}")

# Optional: PCA to 50 dims (similar to Zhuang & Chen '24)
from sklearn.decomposition import PCA
pca = PCA(n_components=50, random_state=cfg["seed"])
XA = pca.fit_transform(XA)
YB = pca.transform(YB)
print("Finished PCA")


In [ ]:
import eval_LowRank
import distance_utils
import monge_rotate_lr
sys.path.insert(0, '../src/HiRef/')
import HiRef

with jax.default_device(jax.devices("gpu")[0]):
    XA = jnp.asarray(XA)
    YB = jnp.asarray(YB)


In [ ]:
from eval_LowRank import evaluate_factors
import run_LowRank
import FRLC.FRLC as frlc

importlib.reload(run_LowRank)
importlib.reload(frlc)
importlib.reload(monge_rotate_lr)

methods = [ "frlc","lot", "monge_conj"]

results = run_LowRank.run_all_methods(
    XA, YB, yA, yB,
    methods=methods,
    rank=cfg["rank"],
    reg=cfg["reg"],
    ot_solver="HiRef",
    device=str(device),
    evaluate_factors_fn=evaluate_factors
)

import json
try:
    import jax.numpy as jnp
    JAX_ARRAY_TYPES = (jnp.ndarray,)
except Exception:
    JAX_ARRAY_TYPES = tuple()

def np_encoder(obj):
    # numpy scalars -> Python scalars
    if isinstance(obj, np.generic):
        return obj.item()
    # numpy arrays or jax arrays -> lists
    if isinstance(obj, (np.ndarray,) + JAX_ARRAY_TYPES):
        return obj.tolist()
    # anything else: let json handle or raise
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

# usage in your loop:
for k, v in results.items():
    if "result" in v:
        print(k, json.dumps(v["result"], indent=2, default=np_encoder))
        if "metrics" in v:
            # optionally drop the big matrix to keep logs short
            metrics_for_log = {kk: vv for kk, vv in v["metrics"].items()
                               if kk != "class_mass_matrix"}
            print(k, "metrics:", json.dumps(metrics_for_log, indent=2, default=np_encoder))
    else:
        print(k, "->", v)
